# FloodVerify CV: evaluation additions (paired bootstrap, building calibration, building-cell agreement)
Runtime: **T4 GPU**. Run top to bottom. About 45-60 min, mostly 6 inference passes over Florence.

Three numbers so far were either untested or flattering:
1. **CE vs Dice + focal**: the seeds showed training randomness does not explain the gap; a **paired bootstrap** on the same tiles tests the data side.
2. **Calibration**: pixel ECE (0.004) was dominated by easy background. Here: ECE on **predicted-building pixels**, the pixels whose probabilities become a cell's `confidence` in the evidence.
3. **Evidence agreement** (86-93%) included empty cells that agree trivially. Here: agreement on **cells whose label has buildings**, plus how often the model invents buildings in empty cells.

Validation only (Florence). Nothing here touches Nepal.

## 0. Setup (only Florence is needed, so no archive extraction)

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # expect Tesla T4

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
OUT = f"{D}/eval_extras"          # results of this notebook, kept on Drive
!mkdir -p "{OUT}"
!ls "{CK}"

Mounted at /content/drive
m0_early_fusion_unet_s0  m1_siamese_unet_ce_s2	m1_siamese_unet_s2
m1_siamese_unet_ce_s0	 m1_siamese_unet_s0	smoke_m1_s0
m1_siamese_unet_ce_s1	 m1_siamese_unet_s1


In [3]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [4]:
# Florence (val) lives in the official "hold" folder: copy that, the manifest and the label evidence. ~3-5 min.
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd
!ls /content/xbd/processed/evidence_labels | wc -l   # expect 10148

hold  processed  rscc_flood_manifest.csv
10148


## 1. The new code
Not on GitHub yet: this cell writes it into the installed package so it can be imported. It will be committed later
as `src/floodsitrep/cv/eval/extras.py` (with tests). Read the docstring at the top: it says what each function does.

In [5]:
%%writefile /content/FloodVerify/src/floodsitrep/cv/eval/extras.py
"""Evaluation additions: paired bootstrap, building-pixel calibration, agreement on building cells.

Owner: member 3. All metrics are PIXEL level unless stated.

1. ``collect``: one pass of a checkpoint over a role. Saves, per tile, the 5x5 pixel confusion
   matrix, and a random sample of PREDICTED-building pixels (logits + true label). Predicted-
   building pixels are the ones whose probabilities become a cell's ``confidence`` in the
   evidence, so that is where calibration matters for the checker.
2. ``paired_bootstrap``: resample the SAME tiles for two runs and recompute the difference in a
   metric each time. Tile difficulty is shared by both runs and cancels out, so this tests "is A
   better than B on this kind of data" more sharply than comparing two separate intervals.
3. ``reliability``: ECE plus the per-bin table (confidence vs accuracy) on the building sample.
4. ``building_cell_agreement``: predicted vs label evidence over cells whose LABEL has at least one
   building (valid_fraction >= 0.5), plus how many label-empty cells the model fills with
   buildings. Empty cells agree trivially, so leaving them out gives the honest number.
"""
import json
from pathlib import Path

import numpy as np

from floodsitrep.cv.eval import metrics as M
from floodsitrep.cv.eval.calibration import ece, softmax


def collect(checkpoint, xbd_root, manifest, out_path, role="val", px_per_tile=1000, seed=0,
            batch_size=8, limit=0, device=None):
    """Run ``checkpoint`` over ``role``; save tile ids, per-tile confusions and a building sample."""
    import torch

    from floodsitrep.cv.eval.infer import iter_predictions
    from floodsitrep.cv.models import load_checkpoint

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, _ = load_checkpoint(checkpoint, device)
    rng = np.random.default_rng(seed)
    ids, cms, lg, lb = [], [], [], []
    for r in iter_predictions(model, xbd_root, manifest, role, batch_size, device=device,
                              limit=limit):
        logits, mask, valid = r["logits"], r["mask"], r["valid"]
        pred = logits.argmax(0)
        ids.append(r["tile_id"])
        cms.append(M.confusion(pred, mask, valid))
        sel = np.flatnonzero((valid & (pred > 0) & (mask <= 4)).ravel())
        if len(sel) > px_per_tile:
            sel = rng.choice(sel, px_per_tile, replace=False)
        lg.append(logits.reshape(logits.shape[0], -1)[:, sel].T.astype(np.float16))
        lb.append(mask.ravel()[sel].astype(np.uint8))
    np.savez_compressed(out_path, tile_ids=np.array(ids), cms=np.stack(cms),
                        bld_logits=np.concatenate(lg), bld_labels=np.concatenate(lb))
    return out_path


def load(path):
    z = np.load(path)
    return {k: z[k] for k in z.files}


def paired_bootstrap(a, b, metric="xview2_pixel_score", n_boot=2000, seed=0):
    """Difference metric(a) - metric(b) over the same resampled tiles.

    a, b: dicts from ``load`` (must cover the same tiles). Returns the point difference, the 95%
    percentile interval, and the share of resamples where the difference is <= 0.
    """
    ia = {t: i for i, t in enumerate(a["tile_ids"])}
    order = [ia[t] for t in b["tile_ids"]]  # align a to b's tile order
    ca, cb = a["cms"][order], b["cms"]
    if len(ca) != len(cb):
        raise ValueError("runs cover different tiles")

    def f(cm):
        return M.flat(M.summarise(cm))[metric]

    point = f(ca.sum(0)) - f(cb.sum(0))
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(cb), len(cb))
        d = f(ca[idx].sum(0)) - f(cb[idx].sum(0))
        if not np.isnan(d):
            diffs.append(d)
    diffs = np.array(diffs)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    return {"metric": metric, "diff": float(point), "lo": float(lo), "hi": float(hi),
            "share_le_0": float((diffs <= 0).mean()), "tiles": len(cb), "n_boot": len(diffs)}


def reliability(logits, labels, t=1.0, n_bins=10):
    """ECE and per-bin table on a pixel sample. Returns (ece, rows)."""
    p = softmax(np.asarray(logits, dtype=np.float64), t)
    conf, pred = p.max(1), p.argmax(1)
    correct = pred == np.asarray(labels)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, n_bins - 1)
    rows = []
    for k in range(n_bins):
        m = idx == k
        if m.any():
            rows.append({"bin": f"{edges[k]:.1f}-{edges[k + 1]:.1f}", "pixels": int(m.sum()),
                         "mean_confidence": float(conf[m].mean()),
                         "accuracy": float(correct[m].mean())})
    return ece(p, labels, n_bins), rows


def building_cell_agreement(pred_dir, label_dir, prefix=""):
    """Agreement over cells whose label has >= 1 building (valid_fraction >= 0.5)."""
    from floodsitrep.common.interfaces import quantifier

    classes = ("none", "minor", "major", "destroyed")

    def qbin(cell, k):
        return "none_present" if cell["buildings"] == 0 else quantifier(cell["class_fraction"][k])

    n = exact = pred_empty = 0
    agree = {k: 0 for k in classes}
    tot_l = tot_p = 0
    empty_label_cells = false_filled = 0
    tiles = 0
    for lp in sorted(Path(label_dir).glob(f"{prefix}*.json")):
        pp = Path(pred_dir) / lp.name
        if not pp.exists():
            continue
        tiles += 1
        lab, prd = json.loads(lp.read_text()), json.loads(pp.read_text())
        for lc, pc in zip(lab["cells"], prd["cells"]):
            if lc.get("valid_fraction", 1.0) < 0.5:
                continue
            if lc["buildings"] == 0:
                empty_label_cells += 1
                false_filled += pc["buildings"] > 0
                continue
            n += 1
            exact += lc["buildings"] == pc["buildings"]
            pred_empty += pc["buildings"] == 0
            tot_l += lc["buildings"]
            tot_p += pc["buildings"]
            for k in classes:
                agree[k] += qbin(lc, k) == qbin(pc, k)
    nan = float("nan")
    return {
        "tiles": tiles,
        "label_cells_with_buildings": n,
        "count_exact_match": exact / n if n else nan,
        "building_ratio_pred_over_label": tot_p / tot_l if tot_l else nan,
        "cells_model_finds_no_building": pred_empty / n if n else nan,
        "quantifier_agreement": {k: v / n if n else nan for k, v in agree.items()},
        "label_empty_cells": empty_label_cells,
        "label_empty_cells_model_fills": (false_filled / empty_label_cells
                                          if empty_label_cells else nan),
    }

Writing /content/FloodVerify/src/floodsitrep/cv/eval/extras.py


In [6]:
import importlib, json, os, numpy as np, pandas as pd
import floodsitrep.cv.eval.extras as EX
from floodsitrep.cv.eval import metrics as M
importlib.reload(EX)
print([f for f in dir(EX) if not f.startswith("_")])

['M', 'Path', 'building_cell_agreement', 'collect', 'ece', 'json', 'load', 'np', 'paired_bootstrap', 'reliability', 'softmax']


In [9]:
!mkdir -p /content/xbd
!tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
!tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
!cp -r "{D}/xbd/tier3" /content/xbd/
!ls /content/xbd
!ls /content/xbd/train/images | grep -c florence

hold  processed  rscc_flood_manifest.csv  test	tier3  train
638


## 2. One inference pass per run (about 6-8 min each on a T4)
Saves per-tile confusion matrices and 1,000 predicted-building pixels per tile to `eval_extras/<run>_val.npz` on Drive.
Already-saved runs are skipped, so after a disconnect just run this cell again. Seed-0 pair first.

In [10]:
RUNS = ["m1_siamese_unet_ce_s0", "m1_siamese_unet_s0",
        "m1_siamese_unet_ce_s1", "m1_siamese_unet_s1",
        "m1_siamese_unet_ce_s2", "m1_siamese_unet_s2"]
for run in RUNS:
    path = f"{OUT}/{run}_val.npz"
    if os.path.exists(path):
        print("skip (exists):", run); continue
    print("collecting", run, flush=True)
    EX.collect(f"{CK}/{run}/best.pt", X, MAN, path, role="val")
!ls -la "{OUT}"

collecting m1_siamese_unet_ce_s0
collecting m1_siamese_unet_s0
collecting m1_siamese_unet_ce_s1
collecting m1_siamese_unet_s1
collecting m1_siamese_unet_ce_s2
collecting m1_siamese_unet_s2
total 68174
drwx------ 2 root root     4096 Nov 17 09:55 .
drwx------ 8 root root     4096 Nov 17 09:17 ..
-rw------- 1 root root   295480 Nov 17 09:52 cloud_screen_val_test.csv
-rw------- 1 root root 11860630 Nov 17 09:36 m1_siamese_unet_ce_s0_val.npz
-rw------- 1 root root 11954690 Nov 17 09:44 m1_siamese_unet_ce_s1_val.npz
-rw------- 1 root root 12056014 Nov 17 09:51 m1_siamese_unet_ce_s2_val.npz
-rw------- 1 root root 11142603 Nov 17 09:40 m1_siamese_unet_s0_val.npz
-rw------- 1 root root 11221688 Nov 17 09:48 m1_siamese_unet_s1_val.npz
-rw------- 1 root root 11268765 Nov 17 09:55 m1_siamese_unet_s2_val.npz


In [11]:
R = {run: EX.load(f"{OUT}/{run}_val.npz") for run in RUNS if os.path.exists(f"{OUT}/{run}_val.npz")}
# Sanity check: each score must equal the best-epoch score in that run's training log
# (CE s0/s1/s2: 0.363 / 0.363 / 0.356; Dice+focal s0/s1/s2: 0.316 / 0.326 / 0.313).
for run, r in R.items():
    score = M.flat(M.summarise(r["cms"].sum(0)))["xview2_pixel_score"]
    print(f"{run:24s} tiles {len(r['tile_ids'])}  score {score:.4f}  building pixels sampled {len(r['bld_labels'])}")

m1_siamese_unet_ce_s0    tiles 2184  score 0.3633  building pixels sampled 1295048
m1_siamese_unet_s0       tiles 2184  score 0.3157  building pixels sampled 1224455
m1_siamese_unet_ce_s1    tiles 2184  score 0.3626  building pixels sampled 1306322
m1_siamese_unet_s1       tiles 2184  score 0.3258  building pixels sampled 1252991
m1_siamese_unet_ce_s2    tiles 2184  score 0.3561  building pixels sampled 1310448
m1_siamese_unet_s2       tiles 2184  score 0.3132  building pixels sampled 1255088


## 3. Paired bootstrap: weighted CE minus Dice + focal, same tiles
2,000 resamples of the 2,184 Florence tiles. For each resample both runs are scored on the identical tiles and the
difference is kept. If the 95% interval stays above 0 (and `share_le_0` is near 0), CE is better on this data, not only
across training seeds.

In [12]:
rows = []
for s in (0, 1, 2):
    a, b = f"m1_siamese_unet_ce_s{s}", f"m1_siamese_unet_s{s}"
    if a in R and b in R:
        for metric in ("xview2_pixel_score", "dmg_f1_destroyed", "loc_f1", "dmg_f1_major"):
            r = EX.paired_bootstrap(R[a], R[b], metric=metric)
            rows.append({"seed": s, "metric": metric, "CE - DiceFocal": r["diff"],
                         "95% lo": r["lo"], "95% hi": r["hi"], "share <= 0": r["share_le_0"]})
pd.DataFrame(rows).round(4)

,seed,metric,CE - DiceFocal,95% lo,95% hi,share <= 0
0,0,xview2_pixel_score,0.0475,-0.0114,0.0807,0.0555
1,0,dmg_f1_destroyed,0.0809,-0.0043,0.1844,0.0420
2,0,loc_f1,-0.0115,-0.0222,-0.0011,0.9860
3,0,dmg_f1_major,0.0238,-0.0239,0.0727,0.1615
4,1,xview2_pixel_score,0.0368,-0.0149,0.0648,0.0725
5,1,dmg_f1_destroyed,0.0785,-0.0047,0.1816,0.0430
6,1,loc_f1,-0.0154,-0.0257,-0.0051,0.9970
7,1,dmg_f1_major,-0.0046,-0.0307,0.0206,0.6595
8,2,xview2_pixel_score,0.0428,-0.0087,0.1051,0.0655
9,2,dmg_f1_destroyed,0.0465,-0.0127,0.1268,0.1020


## 4. Calibration on predicted-building pixels (the Gate 2 model)
Reliability table: pixels grouped by confidence; in a calibrated model `accuracy` is close to `mean_confidence` in
every row. Shown at T = 1 and at the fitted T from `temperature.json`; the Dice + focal run at T = 1 for comparison.

In [13]:
T = json.load(open(f"{CK}/m1_siamese_unet_ce_s0/temperature.json"))["temperature"]
for run, t in [("m1_siamese_unet_ce_s0", 1.0), ("m1_siamese_unet_ce_s0", T), ("m1_siamese_unet_s0", 1.0)]:
    r = R[run]
    e, rows = EX.reliability(r["bld_logits"].astype(np.float64), r["bld_labels"], t=t)
    print(f"\n{run}  T={t:.3f}  building-pixel ECE {e:.4f}  ({len(r['bld_labels'])} pixels)")
    print(pd.DataFrame(rows).round(3).to_string(index=False))


m1_siamese_unet_ce_s0  T=1.000  building-pixel ECE 0.1939  (1295048 pixels)
    bin  pixels  mean_confidence  accuracy
0.2-0.3    3722            0.285     0.065
0.3-0.4   80993            0.365     0.122
0.4-0.5  234690            0.457     0.233
0.5-0.6  296392            0.549     0.311
0.6-0.7  263851            0.649     0.388
0.7-0.8  199713            0.745     0.559
0.8-0.9  160571            0.850     0.844
0.9-1.0   55116            0.927     0.933

m1_siamese_unet_ce_s0  T=1.133  building-pixel ECE 0.1708  (1295048 pixels)
    bin  pixels  mean_confidence  accuracy
0.2-0.3    6227            0.284     0.062
0.3-0.4  112392            0.363     0.130
0.4-0.5  285316            0.456     0.253
0.5-0.6  317671            0.549     0.332
0.6-0.7  261153            0.648     0.430
0.7-0.8  158118            0.747     0.724
0.8-0.9  131696            0.844     0.887
0.9-1.0   22475            0.922     0.955

m1_siamese_unet_s0  T=1.000  building-pixel ECE 0.4013  (1224455 pixels

## 5. Evidence agreement on cells that contain buildings (Gate 2 evidence vs label evidence, Florence)

In [14]:
ag = EX.building_cell_agreement(f"{D}/evidence_predicted/m1_siamese_unet_ce_s0",
                                f"{X}/processed/evidence_labels", prefix="hurricane-florence")
print(json.dumps(ag, indent=2))

{
  "tiles": 2184,
  "label_cells_with_buildings": 4405,
  "count_exact_match": 0.5291713961407492,
  "building_ratio_pred_over_label": 0.9370564064251027,
  "cells_model_finds_no_building": 0.04404086265607265,
  "quantifier_agreement": {
    "none": 0.6699205448354143,
    "minor": 0.5856980703745743,
    "major": 0.7421112372304199,
    "destroyed": 0.8708286038592509
  },
  "label_empty_cells": 14592,
  "label_empty_cells_model_fills": 0.050507127192982455
}


In [15]:
S = pd.read_csv(f"{OUT}/cloud_screen_val_test.csv")
cloudy = set(S[(S.event == "hurricane-florence") & (S.cloud_post > 0.10)].tile)
print(len(cloudy), "cloudy Florence crops dropped")
for run in ["m1_siamese_unet_ce_s0", "m1_siamese_unet_s0"]:
    r = R[run]
    keep = np.array([t not in cloudy for t in r["tile_ids"]])
    a, b = M.flat(M.summarise(r["cms"].sum(0))), M.flat(M.summarise(r["cms"][keep].sum(0)))
    print(run, {k: (round(a[k], 4), round(b[k], 4)) for k in ("xview2_pixel_score", "loc_f1", "dmg_f1_destroyed")})

17 cloudy Florence crops dropped
m1_siamese_unet_ce_s0 {'xview2_pixel_score': (0.3633, 0.3662), 'loc_f1': (0.782, 0.7847), 'dmg_f1_destroyed': (0.124, 0.127)}
m1_siamese_unet_s0 {'xview2_pixel_score': (0.3157, 0.3163), 'loc_f1': (0.7935, 0.7928), 'dmg_f1_destroyed': (0.0431, 0.0436)}


In [16]:
from floodsitrep.cv.eval.calibration import fit_temperature
r = R["m1_siamese_unet_ce_s0"]
z, y = r["bld_logits"].astype(np.float64), r["bld_labels"].astype(int)
idx = np.random.default_rng(0).permutation(len(y))
fit, ev = idx[: len(y) // 2], idx[len(y) // 2 :]          # fit on one half, judge on the other
Tb = fit_temperature(z[fit], y[fit])
for name, t in [("no scaling", 1.0), ("all-pixel T", 1.133), ("building T", Tb)]:
    e, _ = EX.reliability(z[ev], y[ev], t=t)
    print(f"{name:12s} T={t:.3f}  building ECE on held-out half {e:.4f}")

no scaling   T=1.000  building ECE on held-out half 0.1940
all-pixel T  T=1.133  building ECE on held-out half 0.1707
building T   T=1.670  building ECE on held-out half 0.1276


## Done: paste the outputs of sections 3, 4 and 5 in the chat, then stop billing

In [ ]:
from google.colab import runtime
runtime.unassign()